Yuda, 240401010353, IF403

In [9]:
import pandas as pd

# The remote raw URL acting as the API endpoint for the data
url = "https://raw.githubusercontent.com/Geo-y20/Telco-Customer-Churn-/main/Telco%20Customer%20Churn.csv"

# pd.read_csv handles URL API calls automatically
df = pd.read_csv(url)

# Your original analysis steps
print(df.shape)
print(df["Churn"].value_counts(normalize=True))

(7043, 21)
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


In [10]:
from sklearn.model_selection import train_test_split

# 1. Pisahkan X (fitur) dan y (target = Churn)
# Kita drop kolom 'customerID' karena tidak berguna untuk prediksi, dan 'Churn' karena itu targetnya
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

# 2. Encoding fitur kategorikal menggunakan pd.get_dummies
# drop_first=True menghindari jebakan dummy variable (multicollinearity)
X = pd.get_dummies(X, drop_first=True)

# 3. Sekarang jalankan split data tanpa NameError
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Verifikasi hasil split
print(f"X_train shape: {X_tr.shape}")
print(f"X_test shape: {X_te.shape}")

X_train shape: (5634, 6559)
X_test shape: (1409, 6559)


In [11]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
 n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_tr, y_tr)


RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [12]:
from sklearn.metrics import classification_report, roc_auc_score

# 1. Hitung prediksi kelas dan probabilitas untuk data testing
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1] # Ambil probabilitas untuk kelas positif ('Yes')

# 2. Tampilkan classification report
print("=== Classification Report ===")
print(classification_report(y_te, y_pred))

# 3. Tampilkan ROC-AUC Score
roc_auc = roc_auc_score(y_te, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

          No       0.83      0.90      0.86      1035
         Yes       0.64      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score: 0.8301


In [13]:
# 1. Hitung probabilitas churn untuk seluruh data testing
# Kita buat DataFrame baru agar lebih mudah dibaca hasilnya
df_hasil = pd.DataFrame({
    'Actual_Churn': y_te,
    'Probability_Churn': rf.predict_proba(X_te)[:, 1]
})

# Menampilkan 5 sampel teratas hasil prediksi probabilitas
print("=== Sampel Probabilitas Churn ===")
print(df_hasil.head())
print("\n" + "="*30 + "\n")

# 2. Kesimpulan singkat (Silakan sesuaikan nilainya setelah sel 4 dijalankan)
print("=== Kesimpulan ===")
kesimpulan = (
    "Model RandomForest dengan 30 estimators berhasil dilatih dan dievaluasi untuk memprediksi churn pelanggan. "
    "Melalui skor ROC-AUC yang dihasilkan, kita dapat mengukur seberapa baik model dalam membedakan pelanggan yang akan churn dan yang bertahan. "
    "Evaluasi pada classification report memberikan detail metrik precision dan recall, khususnya untuk kelas positif (Churn = Yes). "
    "Hasil probabilitas ini dapat digunakan oleh tim bisnis untuk memberikan intervensi atau promo khusus kepada pelanggan dengan risiko churn tinggi."
)
print(kesimpulan)

=== Sampel Probabilitas Churn ===
     Actual_Churn  Probability_Churn
437            No           0.006667
2280           No           0.656667
2235           No           0.050000
4460           No           0.300000
3761           No           0.020000


=== Kesimpulan ===
Model RandomForest dengan 30 estimators berhasil dilatih dan dievaluasi untuk memprediksi churn pelanggan. Melalui skor ROC-AUC yang dihasilkan, kita dapat mengukur seberapa baik model dalam membedakan pelanggan yang akan churn dan yang bertahan. Evaluasi pada classification report memberikan detail metrik precision dan recall, khususnya untuk kelas positif (Churn = Yes). Hasil probabilitas ini dapat digunakan oleh tim bisnis untuk memberikan intervensi atau promo khusus kepada pelanggan dengan risiko churn tinggi.
